# Loading Data and Defining Models

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Liberation Serif"], # Matches Times New Roman
    "font.size": 10,                    # Standard for most papers
    "axes.labelsize": 10,
    "xtick.labelsize": 8,               
    "ytick.labelsize": 8,
    "legend.fontsize": 9,
    "text.usetex": False,               
    "pdf.fonttype": 42,               
    "ps.fonttype": 42
})
df_deep_agg = pd.read_csv("/kaggle/input/deep-features/deep_agg_features.csv")
df_handcrafted = pd.read_csv("/kaggle/input/handcrafted-features/Handcrafted_Features_flair.csv")
df_fused = pd.merge(df_deep_agg, df_handcrafted, on='id', how='inner')


X = df_fused.drop(columns=['id', 'label'])   # fused deep + radiomic features
y = df_fused['label']                       
print(f"Fused y: {np.bincount(y)}")             

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)


y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred))


importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]
top_features = X.columns[indices][:15]  # Top 15 features from deep + radiomic

plt.figure(figsize=(10,6))
plt.barh(top_features, importances[indices][:15])
plt.title("Top Feature Importances (Random Forest on Fused Features)")
plt.xlabel("Importance Score")
plt.gca().invert_yaxis()
plt.show()


In [ ]:
import shap

# Use the same Random Forest or Fusion Classifier
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

# SHAP Summary Plot
shap.summary_plot(shap_values, X_test, plot_type="bar")
shap.summary_plot(shap_values, X_test)  # Detailed


# SHAP and LIME EXPLAINER

In [ ]:
import shap
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

print("SHAP Plot")

# Fusion Classifier (Logistic Regression)
rf = LogisticRegression(max_iter=10000, solver='lbfgs', C=1.0, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

explainer = shap.LinearExplainer(rf, X_train, feature_dependence="independent")


shap_values = explainer.shap_values(X_test)

# SHAP Summary Plots
shap.summary_plot(shap_values, X_test, plot_type="bar")
shap.summary_plot(shap_values, X_test)                   


In [ ]:
from lime import lime_tabular

explainer = lime_tabular.LimeTabularExplainer(
    training_data=np.array(X_train),
    feature_names=X_train.columns,
    class_names=['Class 0', 'Class 1'],
    mode='classification'
)

sample = X_test.iloc[0]
exp = explainer.explain_instance(sample.values, rf.predict_proba, num_features=10)

exp.show_in_notebook(show_all=False)


# GRAD-CAM visualisation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import numpy as np
import nibabel as nib
import os
from torch.utils.data import Dataset
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define Transforms
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])



train_dir = '/kaggle/input/brain-tumor-segmentation-brats-2019/MICCAI_BraTS_2019_Data_Training'



class Brats2DDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = ['HGG', 'LGG']
        self.samples = []

        # Walk through HGG and LGG folders
        for label in self.classes:
            class_path = os.path.join(root_dir, label)
            for patient_dir in os.listdir(class_path):
                patient_path = os.path.join(class_path, patient_dir)
                if os.path.isdir(patient_path):
                    # Pick one modality, e.g., FLAIR
                    for file in os.listdir(patient_path):
                        if file.endswith('flair.nii.gz') or file.endswith('flair.nii'):
                            self.samples.append((os.path.join(patient_path, file), self.classes.index(label)))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        # Load NIfTI file
        img_nifti = nib.load(path).get_fdata()
        
        # Extract a 2D slice (e.g., the middle slice along the Z-axis)
        slice_idx = img_nifti.shape[2] // 2
        img_slice = img_nifti[:, :, slice_idx]

        # Normalize to 0-255 and convert to 3-channel for ResNet
        img_slice = (img_slice - np.min(img_slice)) / (np.max(img_slice) - np.min(img_slice) + 1e-8)
        img_slice = (img_slice * 255).astype(np.uint8)
        img_slice = np.stack([img_slice] * 3, axis=-1) # Grayscale to RGB

        if self.transform:
            from PIL import Image
            img_slice = Image.fromarray(img_slice)
            img_slice = self.transform(img_slice)

        return img_slice, label

# Replace ImageFolder with the custom class
train_dataset = Brats2DDataset(train_dir, transform=data_transforms)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

model = models.resnet50(pretrained=True)
num_ftrs = model.fc.in_features

model.fc = nn.Linear(num_ftrs, 2) 
model = model.to(device)


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

print("Starting training (HGG vs LGG)...")
epochs = 3

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    if 'train_loader' in locals():
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs} finished. Loss: {running_loss/len(train_loader):.4f}")

print("Training Complete. Model is ready for Grad-CAM.")

In [ ]:
import torch
import torch.nn.functional as F
from torchvision import models, transforms
import numpy as np
import matplotlib.pyplot as plt
import cv2
import nibabel as nib

# Grad-CAM function
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.target_layer = target_layer
        target_layer.register_forward_hook(self.save_activation)
        target_layer.register_backward_hook(self.save_gradient)

    def save_activation(self, module, inp, out):
        self.activation = out

    def save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0]

    def __call__(self, x, class_idx=None):
        output = self.model(x)
        if class_idx is None:
            class_idx = output.argmax().item()

        self.model.zero_grad()
        output[:, class_idx].backward()

        gradients = self.gradients.detach().cpu().numpy()[0]
        activations = self.activation.detach().cpu().numpy()[0]

        weights = np.mean(gradients, axis=(1, 2))
        cam = np.sum(weights[:, np.newaxis, np.newaxis] * activations, axis=0)

        cam = np.maximum(cam, 0)
        cam = cv2.resize(cam, (224, 224))
        cam = cam / cam.max() if cam.max() > 0 else cam
        return cam


nii_path = "/kaggle/input/brain-tumor-segmentation-brats-2019/MICCAI_BraTS_2019_Data_Training/HGG/BraTS19_2013_10_1/BraTS19_2013_10_1_flair.nii" 
nii_img = nib.load(nii_path)
nii_data = nii_img.get_fdata()

slice_idx = nii_data.shape[2] // 2
slice_2d = nii_data[:, :, slice_idx]


slice_2d = (slice_2d - slice_2d.min()) / (slice_2d.max() - slice_2d.min() + 1e-8)


slice_rgb = np.stack([slice_2d]*3, axis=-1)  


slice_resized = cv2.resize(slice_rgb, (224, 224))
transform = transforms.Compose([
    transforms.ToTensor(),
])
img_tensor = transform(slice_resized).unsqueeze(0)
img_tensor = img_tensor.float()

gradcam = GradCAM(model, model.layer4[2].conv3)
heatmap = gradcam(img_tensor)

slice_display = (slice_resized * 255).astype(np.uint8)
heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
overlay = cv2.addWeighted(slice_display, 0.6, heatmap_colored, 0.4, 0)

plt.figure(figsize=(10,5))

plt.subplot(1,3,1)
plt.imshow(slice_2d, cmap='gray')
plt.title("Original .nii Slice")
plt.axis('off')

plt.subplot(1,3,2)
plt.imshow(heatmap, cmap='jet')
plt.title("Grad-CAM Heatmap")
plt.axis('off')

plt.subplot(1,3,3)
plt.imshow(overlay)
plt.title("Grad-CAM on .nii Slice")
plt.axis('off')

plt.tight_layout()
plt.show()
